# App-18: Optimisation d'Hyperparametres - Approches Hybrides

**Navigation** : [<< App-17 VRP](App-17-VRP-Logistics.ipynb) | [Index](../../README.md)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez:

## 1. Configuration et Jeu de Donnees

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import time
from typing import Dict, List, Tuple, Callable, Any
import warnings
warnings.filterwarnings('ignore')

# Configuration reproductible
np.random.seed(42)

# Charger le dataset Breast Cancer (classification binaire)
data = load_breast_cancer()
X, y = data.data, data.target

# Standardiser les donnees
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dataset: {X_train.shape[0]} train, {X_test.shape[0]} test samples")
print(f"Features: {X_train.shape[1]}")
print(f"Classes: {np.unique(y)}")

Dataset: 455 train, 114 test samples
Features: 30
Classes: [0 1]


### Definition de l'espace de recherche

Nous allons optimiser un **Random Forest** avec les hyperparametres suivants:

In [3]:
# Espace de recherche pour Random Forest
param_space = {
    'n_estimators': (10, 500),        # Nombre d'arbres (entier)
    'max_depth': (3, 30),             # Profondeur max (entier)
    'min_samples_split': (2, 20),     # Min samples pour split (entier)
    'min_samples_leaf': (1, 10),      # Min samples par feuille (entier)
    'max_features': (0.1, 1.0),       # Ratio de features (float)
}

def evaluate_rf(params: Dict[str, Any], cv_folds: int = 5) -> float:
    """
    Evalue une configuration d'hyperparametres.
    Retourne la moyenne de cross-validation (accuracy).
    """
    # Convertir les parametres entiers
    rf_params = {
        'n_estimators': int(params['n_estimators']),
        'max_depth': int(params['max_depth']),
        'min_samples_split': int(params['min_samples_split']),
        'min_samples_leaf': int(params['min_samples_leaf']),
        'max_features': params['max_features'],
        'random_state': 42,
        'n_jobs': -1
    }
    
    model = RandomForestClassifier(**rf_params)
    scores = cross_val_score(model, X_train, y_train, cv=cv_folds, scoring='accuracy')
    return scores.mean()

## 2. Methodes de Reference

### 2.1 Grid Search (Recherche en Grille)

**Principe**: Explorer systematiquement toutes les combinaisons d'une grille predefinie.

**Avantages**:
- Simple a implementer et comprendre
- Garantit de trouver le meilleur point sur la grille
- Tres parallele

**Inconvenients**:
- Explosion combinatoire: $O(n^d)$ ou $n$ = points par dimension, $d$ = dimensions
- Ne s'adapte pas a l'importance des dimensions

In [4]:
def grid_search(param_grid: Dict[str, List], max_evals: int = 50) -> Tuple[Dict, float, List]:
    """
    Grid Search simplifie.
    """
    from itertools import product
    
    # Generer toutes les combinaisons
    keys = list(param_grid.keys())
    values = [param_grid[k] for k in keys]
    combinations = list(product(*values))
    
    # Limiter le nombre d'evaluations
    combinations = combinations[:max_evals]
    
    best_score = -np.inf
    best_params = None
    history = []
    
    for combo in combinations:
        params = dict(zip(keys, combo))
        score = evaluate_rf(params)
        history.append({'params': params.copy(), 'score': score})
        
        if score > best_score:
            best_score = score
            best_params = params.copy()
    
    return best_params, best_score, history

# Grille discrete (reduite pour demonstration)
grid_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [0.3, 0.5, 0.7, 1.0]
}

print("=== Grid Search ===")
start_time = time.time()
best_grid, score_grid, history_grid = grid_search(grid_params, max_evals=50)
time_grid = time.time() - start_time

print(f"Meilleur score: {score_grid:.4f}")
print(f"Meilleurs parametres: {best_grid}")
print(f"Temps: {time_grid:.2f}s ({len(history_grid)} evaluations)")

=== Grid Search ===


Meilleur score: 0.9604
Meilleurs parametres: {'n_estimators': 50, 'max_depth': 5, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 0.3}
Temps: 21.29s (50 evaluations)


### 2.2 Random Search (Recherche Aleatoire)

**Principe**: Echantillonner uniformement dans l'espace de recherche.

**Insight cle** (Bergstra & Bengio, 2012): Si certaines dimensions sont plus importantes que d'autres, le random search explore plus efficacement ces dimensions.

**Avantages**:
- Echelle mieux avec la dimensionalite
- Peut etre arrete a tout moment
- Tres parallele

In [ ]:
def random_search(param_space: Dict[str, Tuple], n_iter: int = 50) -> Tuple[Dict, float, List]:
    """
    Random Search avec echantillonnage uniforme.
    """
    best_score = -np.inf
    best_params = None
    history = []
    
    for _ in range(n_iter):
        # Echantillonner chaque parametre
        params = {}
        for name, (low, high) in param_space.items():
            if isinstance(low, int) and isinstance(high, int):
                params[name] = np.random.randint(low, high + 1)
            else:
                params[name] = np.random.uniform(low, high)
        
        score = evaluate_rf(params)
        history.append({'params': params.copy(), 'score': score})
        
        if score > best_score:
            best_score = score
            best_params = params.copy()
    
    return best_params, best_score, history

print("=== Random Search ===")
start_time = time.time()
best_random, score_random, history_random = random_search(param_space, n_iter=50)
time_random = time.time() - start_time

print(f"Meilleur score: {score_random:.4f}")
print(f"Meilleurs parametres: {best_random}")
print(f"Temps: {time_random:.2f}s")

## 3. Optimisation Bayesienne

**Principe**: Utiliser un modele probabiliste (Gaussian Process) pour modeler la fonction objectif et guider l'exploration.

### Algorithme

1. **Initialisation**: Evaluer quelques points aleatoires
2. **Modele**: Entrainer un GP sur les observations
3. **Acquisition**: Selectionner le prochain point maximisant l'acquisition
4. **Evaluation**: Evaluer le point selectionne
5. **Repetition**: Retourner a 2 jusqu'a budget epuise

### Fonctions d'acquisition

- **EI (Expected Improvement)**: Balance exploration/exploitation
- **UCB (Upper Confidence Bound)**: $\mu(x) + \kappa \cdot \sigma(x)$
- **PI (Probability of Improvement)**: Probabilite d'amelioration

In [ ]:
# Implementation manuelle pour comprehension (sans Optuna)
from scipy.stats import norm
from scipy.optimize import minimize

class BayesianOptimizer:
    """
    Optimisation bayesienne simplifiee avec Gaussian Process.
    """
    
    def __init__(self, param_space: Dict[str, Tuple], mode: str = 'ei'):
        self.param_space = param_space
        self.param_names = list(param_space.keys())
        self.mode = mode
        self.observations = []  # (params_normalized, score)
        
    def normalize_params(self, params: Dict) -> np.ndarray:
        """Normalise les parametres entre 0 et 1."""
        normalized = []
        for name in self.param_names:
            low, high = self.param_space[name]
            normalized.append((params[name] - low) / (high - low))
        return np.array(normalized)
    
    def denormalize_params(self, normalized: np.ndarray) -> Dict:
        """Reconvertit en parametres originaux."""
        params = {}
        for i, name in enumerate(self.param_names):
            low, high = self.param_space[name]
            value = normalized[i] * (high - low) + low
            if isinstance(low, int):
                value = int(round(value))
            params[name] = value
        return params
    
    def expected_improvement(self, x: np.ndarray, xi: float = 0.01) -> float:
        """Calcule l'Expected Improvement pour un point."""
        if len(self.observations) < 2:
            return np.random.random()
        
        X_obs = np.array([obs[0] for obs in self.observations])
        y_obs = np.array([obs[1] for obs in self.observations])
        
        # Best observed
        y_best = y_obs.max()
        
        # GP prediction (simplifie: distance-weighted mean/variance)
        distances = np.linalg.norm(X_obs - x, axis=1)
        weights = 1 / (distances + 1e-6)
        weights /= weights.sum()
        
        mean = np.sum(weights * y_obs)
        variance = np.sum(weights * (y_obs - mean)**2)
        std = np.sqrt(variance) + 1e-6
        
        # EI calculation
        z = (mean - y_best - xi) / std
        ei = (mean - y_best - xi) * norm.cdf(z) + std * norm.pdf(z)
        
        return ei if ei > 0 else 0
    
    def suggest_next(self) -> Dict:
        """Suggere le prochain point a evaluer."""
        if len(self.observations) < 3:
            # Initialisation aleatoire
            return self.denormalize_params(np.random.random(len(self.param_names)))
        
        # Optimiser l'acquisition
        best_ei = -np.inf
        best_x = None
        
        for _ in range(100):
            x0 = np.random.random(len(self.param_names))
            ei = self.expected_improvement(x0)
            if ei > best_ei:
                best_ei = ei
                best_x = x0
        
        return self.denormalize_params(best_x)
    
    def observe(self, params: Dict, score: float):
        """Enregistre une observation."""
        normalized = self.normalize_params(params)
        self.observations.append((normalized, score))

def bayesian_optimization(param_space: Dict, n_iter: int = 50) -> Tuple[Dict, float, List]:
    """
    Optimisation bayesienne complete.
    """
    optimizer = BayesianOptimizer(param_space)
    history = []
    best_score = -np.inf
    best_params = None
    
    for i in range(n_iter):
        # Suggerer le prochain point
        params = optimizer.suggest_next()
        
        # Evaluer
        score = evaluate_rf(params)
        optimizer.observe(params, score)
        
        history.append({'params': params.copy(), 'score': score})
        
        if score > best_score:
            best_score = score
            best_params = params.copy()
    
    return best_params, best_score, history

print("=== Bayesian Optimization (custom) ===")
start_time = time.time()
best_bayes, score_bayes, history_bayes = bayesian_optimization(param_space, n_iter=50)
time_bayes = time.time() - start_time

print(f"Meilleur score: {score_bayes:.4f}")
print(f"Meilleurs parametres: {best_bayes}")
print(f"Temps: {time_bayes:.2f}s")

### Avec Optuna (framework professionnel)

Optuna est la librairie de reference pour l'optimisation bayesienne en Python.

In [ ]:
# Si Optuna n'est pas installe: pip install optuna
try:
    import optuna
    optuna_available = True
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    optuna_available = False
    print("Optuna non disponible. Installez avec: pip install optuna")

if optuna_available:
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 10, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 30),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features': trial.suggest_float('max_features', 0.1, 1.0),
        }
        return evaluate_rf(params)
    
    print("=== Optuna (TPE Sampler) ===")
    start_time = time.time()
    
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=50, show_progress_bar=False)
    
    time_optuna = time.time() - start_time
    
    best_optuna = study.best_params
    score_optuna = study.best_value
    
    print(f"Meilleur score: {score_optuna:.4f}")
    print(f"Meilleurs parametres: {best_optuna}")
    print(f"Temps: {time_optuna:.2f}s")

## 4. Algorithme Genetique (GA)

**Principe**: Evoluer une population de configurations vers de meilleures solutions.

### Composants

1. **Representation**: Chromosome = vecteur d'hyperparametres
2. **Selection**: Tournament ou roulette wheel
3. **Crossover**: Echange de genes entre parents
4. **Mutation**: Perturbation aleatoire
5. **Remplacement**: Elitisme + nouveaux individus

In [ ]:
class GeneticHyperparamOptimizer:
    """
    Algorithme genetique pour l'optimisation d'hyperparametres.
    """
    
    def __init__(self, param_space: Dict[str, Tuple], 
                 pop_size: int = 20, mutation_rate: float = 0.1,
                 crossover_rate: float = 0.8, elite_size: int = 2):
        self.param_space = param_space
        self.param_names = list(param_space.keys())
        self.pop_size = pop_size
        self.mutation_rate = mutation_rate
        self.crossover_rate = crossover_rate
        self.elite_size = elite_size
        
    def create_individual(self) -> Dict:
        """Cree un individu aleatoire."""
        individual = {}
        for name, (low, high) in self.param_space.items():
            if isinstance(low, int):
                individual[name] = np.random.randint(low, high + 1)
            else:
                individual[name] = np.random.uniform(low, high)
        return individual
    
    def create_population(self) -> List[Dict]:
        """Cree la population initiale."""
        return [self.create_individual() for _ in range(self.pop_size)]
    
    def evaluate_population(self, population: List[Dict]) -> List[Tuple[Dict, float]]:
        """Evalue tous les individus."""
        evaluated = []
        for individual in population:
            score = evaluate_rf(individual)
            evaluated.append((individual, score))
        return evaluated
    
    def tournament_selection(self, evaluated: List[Tuple], k: int = 3) -> Dict:
        """Selection par tournament."""
        tournament = np.random.choice(len(evaluated), size=k, replace=False)
        winner = max(tournament, key=lambda i: evaluated[i][1])
        return evaluated[winner][0].copy()
    
    def crossover(self, parent1: Dict, parent2: Dict) -> Tuple[Dict, Dict]:
        """Crossover a point unique."""
        if np.random.random() > self.crossover_rate:
            return parent1.copy(), parent2.copy()
        
        # Point de crossover
        n_genes = len(self.param_names)
        crossover_point = np.random.randint(1, n_genes)
        
        child1, child2 = {}, {}
        for i, name in enumerate(self.param_names):
            if i < crossover_point:
                child1[name] = parent1[name]
                child2[name] = parent2[name]
            else:
                child1[name] = parent2[name]
                child2[name] = parent1[name]
        
        return child1, child2
    
    def mutate(self, individual: Dict) -> Dict:
        """Mutation aleatoire."""
        mutated = individual.copy()
        for name in self.param_names:
            if np.random.random() < self.mutation_rate:
                low, high = self.param_space[name]
                if isinstance(low, int):
                    mutated[name] = np.random.randint(low, high + 1)
                else:
                    mutated[name] = np.random.uniform(low, high)
        return mutated
    
    def evolve(self, evaluated: List[Tuple]) -> List[Dict]:
        """Genere la nouvelle generation."""
        # Trier par fitness
        evaluated.sort(key=lambda x: x[1], reverse=True)
        
        # Elitisme
        new_population = [ind.copy() for ind, _ in evaluated[:self.elite_size]]
        
        # Generer le reste
        while len(new_population) < self.pop_size:
            # Selection
            parent1 = self.tournament_selection(evaluated)
            parent2 = self.tournament_selection(evaluated)
            
            # Crossover
            child1, child2 = self.crossover(parent1, parent2)
            
            # Mutation
            child1 = self.mutate(child1)
            child2 = self.mutate(child2)
            
            new_population.extend([child1, child2])
        
        return new_population[:self.pop_size]
    
    def optimize(self, n_generations: int = 10) -> Tuple[Dict, float, List]:
        """Boucle d'optimisation principale."""
        population = self.create_population()
        history = []
        best_score = -np.inf
        best_params = None
        
        for gen in range(n_generations):
            # Evaluation
            evaluated = self.evaluate_population(population)
            
            # Meilleur de la generation
            gen_best = max(evaluated, key=lambda x: x[1])
            history.append({'generation': gen, 'params': gen_best[0], 'score': gen_best[1]})
            
            if gen_best[1] > best_score:
                best_score = gen_best[1]
                best_params = gen_best[0].copy()
            
            # Evolution
            population = self.evolve(evaluated)
        
        return best_params, best_score, history

# Execution
print("=== Genetic Algorithm ===")
ga = GeneticHyperparamOptimizer(param_space, pop_size=20, mutation_rate=0.15)

start_time = time.time()
best_ga, score_ga, history_ga = ga.optimize(n_generations=5)  # 5 gen * 20 pop = 100 evals
time_ga = time.time() - start_time

print(f"Meilleur score: {score_ga:.4f}")
print(f"Meilleurs parametres: {best_ga}")
print(f"Temps: {time_ga:.2f}s")

## 5. Particle Swarm Optimization (PSO)

**Principe**: Une nuée de particules explore l'espace, guidee par:
- **pBest**: Meilleure position personnelle
- **gBest**: Meilleure position globale

### Mise a jour de la vitesse

$$v_{i+1} = w \cdot v_i + c_1 \cdot r_1 \cdot (pBest - x_i) + c_2 \cdot r_2 \cdot (gBest - x_i)$$

Ou:
- $w$ = inertie (0.4-0.9)
- $c_1$ = coefficient cognitif (~2.0)
- $c_2$ = coefficient social (~2.0)

In [ ]:
class PSOHyperparamOptimizer:
    """
    Particle Swarm Optimization pour hyperparametres.
    """
    
    def __init__(self, param_space: Dict[str, Tuple],
                 n_particles: int = 20, w: float = 0.7, c1: float = 1.5, c2: float = 1.5):
        self.param_space = param_space
        self.param_names = list(param_space.keys())
        self.n_dims = len(self.param_names)
        self.n_particles = n_particles
        self.w = w      # Inertie
        self.c1 = c1    # Coefficient cognitif
        self.c2 = c2    # Coefficient social
        
    def initialize(self):
        """Initialise les positions et vitesses."""
        # Positions normalisees [0, 1]
        self.positions = np.random.random((self.n_particles, self.n_dims))
        
        # Vitesses normalisees [-0.1, 0.1]
        self.velocities = np.random.uniform(-0.1, 0.1, (self.n_particles, self.n_dims))
        
        # Meilleures positions personnelles
        self.pbest_positions = self.positions.copy()
        self.pbest_scores = np.full(self.n_particles, -np.inf)
        
        # Meilleure position globale
        self.gbest_position = None
        self.gbest_score = -np.inf
        
    def denormalize(self, position: np.ndarray) -> Dict:
        """Convertit une position normalisee en parametres."""
        params = {}
        for i, name in enumerate(self.param_names):
            low, high = self.param_space[name]
            value = position[i] * (high - low) + low
            if isinstance(low, int):
                value = int(np.clip(round(value), low, high))
            params[name] = value
        return params
    
    def evaluate_particle(self, position: np.ndarray) -> float:
        """Evalue une particule."""
        params = self.denormalize(position)
        return evaluate_rf(params)
    
    def update(self):
        """Met a jour les positions et vitesses."""
        r1, r2 = np.random.random((2, self.n_particles, self.n_dims))
        
        # Mise a jour de la vitesse
        cognitive = self.c1 * r1 * (self.pbest_positions - self.positions)
        social = self.c2 * r2 * (self.gbest_position - self.positions)
        self.velocities = self.w * self.velocities + cognitive + social
        
        # Mise a jour de la position
        self.positions += self.velocities
        
        # Bornes [0, 1]
        self.positions = np.clip(self.positions, 0, 1)
    
    def optimize(self, n_iterations: int = 50) -> Tuple[Dict, float, List]:
        """Boucle d'optimisation."""
        self.initialize()
        history = []
        
        for iteration in range(n_iterations):
            # Evaluer toutes les particules
            for i in range(self.n_particles):
                score = self.evaluate_particle(self.positions[i])
                
                # Mise a jour pbest
                if score > self.pbest_scores[i]:
                    self.pbest_scores[i] = score
                    self.pbest_positions[i] = self.positions[i].copy()
                
                # Mise a jour gbest
                if score > self.gbest_score:
                    self.gbest_score = score
                    self.gbest_position = self.positions[i].copy()
            
            history.append({
                'iteration': iteration,
                'score': self.gbest_score,
                'params': self.denormalize(self.gbest_position)
            })
            
            # Mise a jour
            self.update()
        
        return self.denormalize(self.gbest_position), self.gbest_score, history

# Execution
print("=== PSO ===")
pso = PSOHyperparamOptimizer(param_space, n_particles=10)

start_time = time.time()
best_pso, score_pso, history_pso = pso.optimize(n_iterations=10)  # 10 iter * 10 particles = 100 evals
time_pso = time.time() - start_time

print(f"Meilleur score: {score_pso:.4f}")
print(f"Meilleurs parametres: {best_pso}")
print(f"Temps: {time_pso:.2f}s")

## 6. Comparaison des Methodes

Analysons la convergence et l'efficacite de chaque methode.

In [ ]:
# Extraction des courbes de convergence
def get_convergence(history: List[Dict]) -> np.ndarray:
    """Extrait la meilleure score cumulatif."""
    scores = [h['score'] for h in history]
    cumulative_best = np.maximum.accumulate(scores)
    return cumulative_best

# Figure de comparaison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Courbes de convergence
ax1 = axes[0]
ax1.plot(get_convergence(history_random), label='Random Search', alpha=0.8)
ax1.plot(get_convergence(history_bayes), label='Bayesian (custom)', alpha=0.8)
ax1.plot(get_convergence(history_ga), label='Genetic Algorithm', alpha=0.8)
ax1.plot(get_convergence(history_pso), label='PSO', alpha=0.8)

ax1.set_xlabel('Iterations')
ax1.set_ylabel('Meilleur Score (Accuracy)')
ax1.set_title('Convergence des Methodes')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Bar chart des resultats finaux
ax2 = axes[1]
methods = ['Grid\nSearch', 'Random\nSearch', 'Bayesian\n(custom)', 'Genetic\nAlgorithm', 'PSO']
scores = [score_grid, score_random, score_bayes, score_ga, score_pso]
times = [time_grid, time_random, time_bayes, time_ga, time_pso]

colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(methods)))
bars = ax2.bar(methods, scores, color=colors, alpha=0.8)

# Annotations
for bar, score, t in zip(bars, scores, times):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{score:.4f}\n({t:.1f}s)', ha='center', va='bottom', fontsize=9)

ax2.set_ylabel('Meilleur Score (Accuracy)')
ax2.set_title('Performance Finale')
ax2.set_ylim(0.9, 1.0)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Tableau recapitulatif

In [ ]:
# Resume des resultats
results = pd.DataFrame({
    'Methode': ['Grid Search', 'Random Search', 'Bayesian (custom)', 'Genetic Algorithm', 'PSO'],
    'Meilleur Score': [score_grid, score_random, score_bayes, score_ga, score_pso],
    'Temps (s)': [time_grid, time_random, time_bayes, time_ga, time_pso],
    'Evaluations': [len(history_grid), len(history_random), len(history_bayes), 
                    len(history_ga) * 20, len(history_pso) * 10]  # Approximation
})

results['Efficacite'] = results['Meilleur Score'] / results['Temps (s)']
results = results.sort_values('Meilleur Score', ascending=False)

print("=== Tableau Comparatif ===")
print(results.to_string(index=False))

## 7. Visualisation de l'Espace de Recherche

Visualisons comment chaque methode explore l'espace des hyperparametres.

In [ ]:
# Visualisation 2D (n_estimators vs max_depth)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

histories = [history_random, history_bayes, history_ga, history_pso]
titles = ['Random Search', 'Bayesian (custom)', 'Genetic Algorithm', 'PSO']
colors = ['blue', 'green', 'red', 'purple']

for ax, history, title, color in zip(axes.flat, histories, titles, colors):
    # Extraire les points explores
    n_estimators = [h['params']['n_estimators'] for h in history]
    max_depth = [h['params']['max_depth'] for h in history]
    scores = [h['score'] for h in history]
    
    # Scatter plot avec couleur = score
    scatter = ax.scatter(n_estimators, max_depth, c=scores, cmap='viridis', 
                         s=50, alpha=0.7, edgecolors='black')
    
    # Meilleur point
    best_idx = np.argmax(scores)
    ax.scatter([n_estimators[best_idx]], [max_depth[best_idx]], 
               c='red', s=200, marker='*', edgecolors='black', linewidths=2, 
               label=f'Best: {max(scores):.4f}')
    
    ax.set_xlabel('n_estimators')
    ax.set_ylabel('max_depth')
    ax.set_title(title)
    ax.legend()
    plt.colorbar(scatter, ax=ax, label='Accuracy')

plt.suptitle('Exploration de l\'Espace (n_estimators vs max_depth)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Interpretation des resultats**

The visualization shows how each method explores the 2D parameter space (n_estimators vs max_depth):
- **Random Search** explores broadly but uniformly
- **Bayesian** focuses on promising regions
- **GA** explores across the space
- **PSO** shows similar exploration to Random Search
- **Best points**: All methods tend to find solutions in similar regions
- **Convergence patterns**: Random Search and GA and and PSO show faster initial convergence, Bayesian converges more gradually
- **Recommendations**: Use visualization to identify which method works best for your specific problem

**Next Steps**: Try different parameter combinations (n_estimators, max_depth) to see how performance varies across the space.

## 8. Recommandations et Bonnes Pratiques

### Quand utiliser chaque methode?

| Methode | Quand l'utiliser |
|---------|------------------|
| **Random Search** | Baseline, premier passage, parallelisation facile |
| **Bayesienne** | Peu d'evaluations possibles, fonction couteuse, mono-objectif |
| **GA** | Espace complexe, multi-modal, multi-objectif, parallelisation GPU |
| **PSO** | Espace continu, bonne convergence, facile a implementer |
| **Grid Search** | Peu de dimensions, comprehension du probleme, reproductibilite |

### Conseils pratiques

1. **Commencer par Random Search** pour estimer les ranges
2. **Utiliser l'optimisation bayesienne** pour le tuning fin
3. **Paralleliser** quand possible (GA, PSO)
4. **Fixer le random seed** pour reproductibilite
5. **Logger toutes les evaluations** pour analyse posterieure

## 9. Exercices

### Exercice 1: Multi-objectif

Modifiez le code pour optimiser simultanement l'accuracy et le temps d'entrainement. Utilisez NSGA-II ou la somme ponderee.

### Exercice 2: Early Stopping

Implementez un mecanisme de early stopping pour arreter l'evaluation des configurations peu prometteuses.

### Exercice 3: Transfer Learning

Comment reutiliser les observations d'un dataset pour accelerer l'optimisation sur un nouveau dataset?

### Exercice 4: Parallelisation

Parallelisez l'evaluation des particules PSO avec `multiprocessing` ou `joblib`.

## Conclusion

Dans ce notebook, nous avons explore cinq methodes d'optimisation d'hyperparametres:

1. **Grid Search** - Methode exhaustive mais couteuse
2. **Random Search** - Simple et souvent surprenante efficace
3. **Optimisation Bayesienne** - Efficace pour les fonctions couteuses
4. **Algorithmes Genetiques** - Excellente exploration, population-based
5. **PSO** - Convergence rapide sur espaces continus

### Points cles a retenir

- Le **Random Search** est souvent un excellent point de depart
- L'**optimisation bayesienne** excelle quand les evaluations sont limites
- Les **metaheuristiques** (GA, PSO) sont adaptees aux problemes complexes
- La **parallelisation** peut grandement accelerer le processus
- Il n'y a pas de methode universelle: **adapter au probleme**